<a href="https://colab.research.google.com/github/ahmedsaad-tech/TimeSeriesAnalysisWithPython/blob/master/tools/grids/grids.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install anemoi-datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 1.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of eccodeslib to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 440.0/440.0 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.3/211.3 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
from anemoi.datasets import open_dataset

In [7]:
import yaml

with open('/content/grids1.yaml', 'r') as file:
    grids1_config = yaml.safe_load(file)

print(grids1_config)

{'common': {'mars_request': {'expver': '0001', 'grid': '1/1'}}, 'dates': {'start': datetime.datetime(2024, 1, 1, 0, 0), 'end': datetime.datetime(2024, 1, 1, 18, 0), 'frequency': '6h'}, 'input': {'join': [{'mars': {'expver': '0001', 'grid': '1/1', 'param': ['2t', '10u', '10v', 'lsm'], 'levtype': 'sfc', 'stream': 'oper', 'type': 'an'}}, {'mars': {'expver': '0001', 'grid': '1/1', 'param': ['q', 't', 'z'], 'levtype': 'pl', 'level': [50, 100], 'stream': 'oper', 'type': 'an'}}, {'accumulations': {'expver': '0001', 'grid': '1/1', 'levtype': 'sfc', 'param': ['cp', 'tp']}}, {'forcings': {'template': '${input.join.0.mars}', 'param': ['cos_latitude', 'sin_latitude']}}]}, 'output': {'order_by': ['valid_datetime', 'param_level', 'number'], 'remapping': {'param_level': '{param}_{levelist}'}, 'statistics': 'param_level'}}


In [22]:
import anemoi.datasets.usage.gridded

# ds_obj = anemoi.datasets.usage.dataset.Dataset()
print(dir(anemoi.datasets.usage.gridded))
# anemoi.datasets.usage.dataset.Dataset(grids1_config, "grids.zarr")
# print("grids.zarr created successfully.")

['Any', 'DEBUG_ZARR_LOADING', 'Dataset', 'DebugStore', 'HTTPStore', 'LOG', 'NDArray', 'Node', 'QUIET', 'S3Store', 'Shape', 'Source', 'TupleIndex', 'ZarrFileNotFoundError', 'ZarrStore', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'cached_property', 'collect_analytics', 'dataset_lookup', 'datetime', 'frequency_to_timedelta', 'load_config', 'logging', 'name_to_zarr_store', 'np', 'open_zarr', 'open_zarr_store', 'os', 'tempfile', 'urlparse', 'zarr']


In [14]:
import anemoi.datasets
print(dir(anemoi.datasets))

['MissingDateError', '__all__', '__annotations__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_version', 'add_dataset_path', 'add_named_dataset', 'check_dataset_name', 'compat', 'create', 'list_dataset_names', 'open_dataset', 'usage']


In [4]:
def plot_grid(ds, path, s=0.1, c="r", grids=None, point=None):
    import matplotlib.pyplot as plt
    import cartopy.crs as ccrs
    import numpy as np

    lats, lons = ds.latitudes, ds.longitudes

    fig = plt.figure(figsize=(9, 9))
    proj = ccrs.NearsidePerspective(
        central_latitude=50.0, central_longitude=-25.0, satellite_height=4e6
    )

    ax = plt.axes(projection=proj)

    def fill():
        # Make sure we have a full globe
        lons, lats = np.meshgrid(np.arange(-180, 180, 1), np.arange(-90, 90, 1))
        x, y, _ = proj.transform_points(
            ccrs.PlateCarree(), lons.flatten(), lats.flatten()
        ).T

        mask = np.invert(np.logical_or(np.isinf(x), np.isinf(y)))
        x = np.compress(mask, x)
        y = np.compress(mask, y)

        # ax.tricontourf(x, y, values)
        ax.scatter(x, y, s=0, c="w")

    fill()

    def plot(what, s, c):
        x, y, _ = proj.transform_points(ccrs.PlateCarree(), lons[what], lats[what]).T

        mask = np.invert(np.logical_or(np.isinf(x), np.isinf(y)))
        x = np.compress(mask, x)
        y = np.compress(mask, y)

        # ax.tricontourf(x, y, values)
        ax.scatter(x, y, s=s, c=c)

    if grids:
        a = 0
        for i, b in enumerate(grids):
            if s[i] is not None:
                plot(slice(a, b), s[i], c[i])
            a += b
    else:
        plot(..., s, c)

    if point:
        point = np.array(point, dtype=np.float64)
        x, y, _ = proj.transform_points(ccrs.PlateCarree(), point[1], point[0]).T
        ax.scatter(x, y, s=100, c="k")

    ax.coastlines()

    if isinstance(path, str):
        fig.savefig(path, bbox_inches="tight")
    else:
        for p in path:
            fig.savefig(p, bbox_inches="tight")

In [6]:
ds = open_dataset("grids1.zarr")

PathNotFoundError: nothing found at path 'grids1.zarr'

In [ ]:
plot_grid(ds, ["thinning-before.png", "cutout-1.png"], s=0.5)

In [ ]:
ds = open_dataset(ds, thinning=4)

In [ ]:
plot_grid(ds, "thinning-after.png", s=1, c="b")

In [ ]:
ds = open_dataset(cutout=["grids2.zarr", "grids1.zarr"])


plot_grid(
    ds,
    "cutout-4.png",
    s=[0.5, 0.5],
    grids=ds.grids,
    c=["g", "r"],
)

In [ ]:
plot_grid(ds, "cutout-2.png", s=[0.5, None], grids=ds.grids, c=["g", "r"])

In [ ]:
plot_grid(ds, "cutout-3.png", s=[None, 0.5], grids=ds.grids, c=["g", "r"])

In [ ]:
ds = open_dataset("grids1.zarr")
ds = open_dataset(ds, area=(60, -50, 20, 0))
plot_grid(ds, "area-1.png", s=0.5)